# Notebook 09
# PneumoXNet: Proposed Model for Multi-Class Pneumonia Classification

This notebook implements the proposed deep learning architecture, PneumoXNet, for multi-class pneumonia classification using chest X-ray images.

The proposed model is built upon EfficientNet-B2 and incorporates:
- CBAM (Convolutional Block Attention Module)
- Multi-Scale Feature Fusion
- Grad-CAM Explainability

The objective is to improve classification performance while providing better model interpretability.

In [1]:
# ============================================================
# Cell 1: Import Required Libraries
# ============================================================

# Standard Library
import copy
import random
import time
from pathlib import Path

# Data Processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress Bar
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Pretrained Model
from torchvision.models import (
    efficientnet_b2,
    EfficientNet_B2_Weights
)

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.utils.class_weight import compute_class_weight

# Plot Style
sns.set_theme(
    style="whitegrid",
    context="notebook"
)

print("=" * 70)
print("Cell 1 : Required Libraries")
print(f"PyTorch Version : {torch.__version__}")
print("=" * 70)

Cell 1 : Required Libraries
PyTorch Version : 2.13.0+cu126


## Objectives

- Load the processed chest X-ray dataset
- Build the proposed PneumoXNet architecture
- Integrate CBAM attention mechanism
- Perform multi-scale feature fusion
- Train and validate the proposed model
- Evaluate on the independent test set
- Generate Grad-CAM visualizations
- Compare the proposed model with baseline models

In [2]:
# ============================================================
# Cell 2: Device Configuration
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Cell 2 : Device Configuration")
print("-" * 70)

print(f"Selected Device : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version : {torch.version.cuda}")

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(f"GPU Memory   : {gpu_memory:.2f} GB")

else:
    print("Running on CPU")

print("=" * 70)

Cell 2 : Device Configuration
----------------------------------------------------------------------
Selected Device : cuda
GPU Name     : NVIDIA GeForce RTX 3050
CUDA Version : 12.6
GPU Memory   : 8.00 GB


## Workflow

1. Import Required Libraries
2. Device Configuration
3. Random Seed Initialization
4. Hyperparameter Configuration
5. Project Directory Configuration
6. Dataset Preparation
7. DataLoader Creation
8. Class Weight Computation
9. CBAM Module
10. Multi-Scale Feature Fusion Module
11. PneumoXNet Architecture
12. Model Training
13. Model Evaluation
14. Grad-CAM Visualization
15. Performance Comparison

In [3]:
# ============================================================
# Cell 3: Random Seed Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 70)
print("Cell 3 : Random Seed Configuration")
print("-" * 70)
print(f"Random Seed : {SEED}")
print("Reproducibility : Enabled")
print("=" * 70)

Cell 3 : Random Seed Configuration
----------------------------------------------------------------------
Random Seed : 42
Reproducibility : Enabled


In [4]:
# ============================================================
# Cell 4: Hyperparameter Configuration
# ============================================================

# Experiment Information

MODEL_NAME = "PneumoXNet"

MODEL_ID = "pneumoxnet"

MODEL_VERSION = "Proposed"

EXPERIMENT_NAME = "PneumoXNet_Proposed"

# Hyperparameters

IMAGE_SIZE = 224

BATCH_SIZE = 16

EPOCHS = 30

LEARNING_RATE = 1e-4

NUM_CLASSES = 3

NUM_WORKERS = 0

CLASS_NAMES = [
    "BACTERIA",
    "NORMAL",
    "VIRUS"
]

print("=" * 70)
print("Cell 4 : Hyperparameter Configuration")
print("-" * 70)

print(f"Model Name      : {MODEL_NAME}")
print(f"Model ID        : {MODEL_ID}")
print(f"Version         : {MODEL_VERSION}")
print(f"Experiment      : {EXPERIMENT_NAME}")
print(f"Image Size      : {IMAGE_SIZE}")
print(f"Batch Size      : {BATCH_SIZE}")
print(f"Epochs          : {EPOCHS}")
print(f"Learning Rate   : {LEARNING_RATE}")
print(f"Number Classes  : {NUM_CLASSES}")

print("=" * 70)

Cell 4 : Hyperparameter Configuration
----------------------------------------------------------------------
Model Name      : PneumoXNet
Model ID        : pneumoxnet
Version         : Proposed
Experiment      : PneumoXNet_Proposed
Image Size      : 224
Batch Size      : 16
Epochs          : 30
Learning Rate   : 0.0001
Number Classes  : 3
